# Apply Work-Author Curations

Applies work-author curations from `openalex.works.work_author_curation_{claims,removals}` onto `openalex.works.work_authors`. Runs after `Author_Matching` and before `UpdateWorkAuthorships`.

## Apply-every-cycle semantics (new design 2026-05-11)

Apply runs every cycle as a post-`MatchAuthors` override — no apply-once gating, no `applied_at`, no per-row state tracking, no pending-snapshot tables. Both MERGEs are idempotent: after the first cycle changes a slot, subsequent cycles find the same target state and the no-op churn guards keep the UPDATE from firing.

This deliberately inverts the prior apply-once design (shipped 2026-05-08, walden a9d067d): the simpler invariant is "apply is the single source of truth for curated state, every cycle." Trade-offs: ES re-flow churn rises (every curated work_id enters the pending-sync queue every cycle), and the apply task is now load-bearing for curated-state correctness — a skipped or failed apply silently reverts curated state to whatever `MatchAuthors` last wrote. See oxjob #168 PLAN.md design-pivot block for full rationale.

## Semantics

- **Claim** (name-anchored, was `add`): MERGE on `(work_id, raw_author_name)` from `work_author_claim_curations`, deduped latest-wins (`created DESC, curation_id DESC`) per `(work_id, raw_author_name)` — conflicting claims on the same slot occur in practice and the most recent curation is the operative one. The curator's UI signal is "this printed name on the paper is me," and the source row's `raw_author_name` (extracted from the property string by the Postgres view) matches `work_authors.raw_author_name` exactly. No-op churn guard via `WHEN MATCHED AND (target.author_id IS NULL OR target.author_id <> source.author_id)` so already-curated slots don't bump `updated_at` every cycle.
- **Remove** (sticky disclaim, every cycle): MERGE on `(work_id, author_id)` from `work_author_remove_curations` always sets `author_id = NULL`. **There is no block-list in `MatchAuthors`** — disclaim is enforced solely by this notebook running after `MatchAuthors` and re-NULLing the slot every cycle. Naturally idempotent: once a row is NULLed, the JOIN on `target.author_id = source.author_id` stops matching, so subsequent applies are no-ops until `MatchAuthors` re-attaches the pair (at which point the next apply re-NULLs it).

## Affected works queue

Every curated work_id (claims + removals) gets inserted into `openalex.works.curated_work_ids_pending_sync` every cycle. `UpdateWorkAuthorships` picks them up and truncates the queue at the end of its run. This is the apply-every-cycle ES churn trade-off — fine at current volumes (hundreds of curations); revisit if it materially impacts queue depth, `UpdateWorkAuthorships` runtime, or `sync_works` throughput.

## Apply claim curations to work_authors

Name-anchored MERGE on `(work_id, raw_author_name)` from `work_author_claim_curations`, deduped latest-wins per slot (conflicting claims on the same slot are real — see the cell comment). The no-op churn guard skips the UPDATE when the slot already holds the curated author — without it, every cycle would bump `updated_at` on every curated slot for zero behavioral change.

In [ ]:
%sql
-- Latest-wins dedup: multiple users (or one user double-submitting) can claim the
-- same (work_id, raw_author_name) slot with DIFFERENT author_ids — conflicting
-- curation rows that all map to one target slot. Plain DISTINCT can't collapse
-- them (the remove-side trick) because author_id differs; without dedup the MERGE
-- fails DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW. The most recent claim
-- (created DESC) is the operative one; curation_id breaks exact-timestamp ties.
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT work_id, raw_author_name, author_id
  FROM openalex.works.work_author_claim_curations
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY work_id, raw_author_name
    ORDER BY created DESC, curation_id DESC
  ) = 1
) AS source
ON target.work_id = source.work_id
   AND target.raw_author_name = source.raw_author_name
WHEN MATCHED AND (target.author_id IS NULL OR target.author_id <> source.author_id) THEN
  UPDATE SET
    target.author_id  = source.author_id,
    target.updated_at = current_timestamp()

## Apply remove curations to work_authors

NULL the `author_id` at any `(work_id, author_sequence)` where it currently equals the disclaimed author. Naturally idempotent — once NULLed, the `target.author_id = source.author_id` JOIN stops matching, so subsequent applies no-op until `MatchAuthors` re-attaches the pair.

In [ ]:
%sql
-- DISTINCT collapses the legitimate many-curations -> one-slot fan-in:
-- multiple users can disclaim the same (work_id, author_id), producing several
-- valid curation rows (distinct curation_id) that all map to the same target slot.
-- Without DISTINCT the MERGE fails DELTA_MULTIPLE_SOURCE_ROW_MATCHING_TARGET_ROW.
-- The UPDATE sets a constant (NULL), so every colliding row is identical -> safe to collapse.
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT DISTINCT work_id, author_id
  FROM openalex.works.work_author_remove_curations
) AS source
ON target.work_id = source.work_id
   AND target.author_id = source.author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id  = NULL,
    target.updated_at = current_timestamp()

## Queue affected work_ids for re-sync

Insert every curated work_id (claims + removals) into `openalex.works.curated_work_ids_pending_sync` every cycle. `UpdateWorkAuthorships` picks them up and truncates the queue at the end of its run. This is the apply-every-cycle ES churn trade-off (see header).

In [ ]:
%sql
INSERT INTO openalex.works.curated_work_ids_pending_sync
SELECT DISTINCT work_id, current_timestamp() AS added_datetime
FROM (
    SELECT work_id FROM openalex.works.work_author_claim_curations
    UNION
    SELECT work_id FROM openalex.works.work_author_remove_curations
)

## Verify

In [ ]:
%sql
-- Totals only (apply runs every cycle, so "applied this run" is the same as "total").
SELECT
    (SELECT COUNT(*) FROM openalex.works.work_author_claim_curations)   AS claim_curations_total,
    (SELECT COUNT(*) FROM openalex.works.work_author_remove_curations) AS removal_curations_total,
    (SELECT COUNT(DISTINCT work_id) FROM (
        SELECT work_id FROM openalex.works.work_author_claim_curations
        UNION
        SELECT work_id FROM openalex.works.work_author_remove_curations
    ))                                                                  AS distinct_curated_works

In [ ]:
%sql
-- Spot-check work_authors for everything touched by curations.
-- Claim slots should show curated author_id at the claimed (work_id, raw_author_name);
-- removed slots should show author_id IS NULL at the disclaimed (work_id, author_id).
SELECT wa.work_id, wa.author_sequence, wa.author_id, wa.raw_author_name, wa.updated_at
FROM openalex.works.work_authors wa
WHERE wa.work_id IN (
    SELECT work_id FROM openalex.works.work_author_claim_curations
    UNION
    SELECT work_id FROM openalex.works.work_author_remove_curations
)
ORDER BY wa.updated_at DESC
LIMIT 100